# Stage 5 — LoRA seed-42 pilot

Runs, in order: (1) a matched three-step Stage 3 full-fine-tuning memory profile, (2) the three-step LoRA smoke test at batch 32, and only after it passes, (3) the seed-42 LoRA pilot. Validation only; the SciFact test split is never loaded. All outputs persist under `MyDrive/RAFT/stage5_lora`.


In [ ]:
%pip uninstall -y -q torchao
%pip install -q "sentence-transformers==6.0.1" "transformers==5.16.1" "faiss-cpu==1.15.0" datasets pandas peft


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import base64
import hashlib
import io
import zipfile
from pathlib import Path

PAYLOAD_SHA256 = '090C0CC6E7D190E28434776A986E5DEDE62AC2D5AACCFC256DFB3ED0A44D9215'
PAYLOAD_B64 = '...'
payload = base64.b64decode(PAYLOAD_B64)
assert hashlib.sha256(payload).hexdigest().upper() == PAYLOAD_SHA256
SOURCE_DIR = Path('/content/drive/MyDrive/RAFT/stage5_lora/source_snapshot')
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    assert all(not Path(name).is_absolute() and '..' not in Path(name).parts for name in archive.namelist())
    archive.extractall(SOURCE_DIR)
print(f'Stage 5 source ready: {SOURCE_DIR}')
print(f'PAYLOAD_SHA256={PAYLOAD_SHA256}')


In [ ]:
import json
import os
import subprocess
import sys
import torch

DRIVE_ROOT = Path('/content/drive/MyDrive/RAFT')
os.environ['HF_HOME'] = str(DRIVE_ROOT / 'huggingface_cache')
assert torch.cuda.is_available(), 'Select a free Colab GPU runtime before continuing.'
required = [
    DRIVE_ROOT / 'stage1_scifact/processed/corpus.jsonl',
    DRIVE_ROOT / 'stage1_scifact/processed/queries_train.jsonl',
    DRIVE_ROOT / 'stage1_scifact/processed/qrels_train.tsv',
    DRIVE_ROOT / 'stage1_scifact/processed/queries_validation.jsonl',
    DRIVE_ROOT / 'stage1_scifact/processed/qrels_validation.tsv',
    DRIVE_ROOT / 'stage3_inbatch/final_seed42/results/stage3_final_result.json',
    DRIVE_ROOT / 'stage3_inbatch/final_seed42/best_checkpoint',
    SOURCE_DIR / 'results/stage3_final_summary.json',
    SOURCE_DIR / 'results/stage4_final_summary.json',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, f'Missing restartable Drive inputs: {missing}'
config = json.loads((SOURCE_DIR / 'configs/stage5_lora_pilot.json').read_text())
print(json.dumps(config, indent=2))


In [ ]:
completed = subprocess.run([
    sys.executable,
    str(SOURCE_DIR / 'src/run_stage5_pilot.py'),
    '--repo-dir', str(SOURCE_DIR),
    '--drive-root', str(DRIVE_ROOT),
], check=True)
print(completed)


In [ ]:
import pandas as pd

RESULTS = DRIVE_ROOT / 'stage5_lora/pilot_seed42/results'
comparison = pd.read_csv(RESULTS / 'stage5_pilot_validation_comparison.csv')
efficiency = json.loads((RESULTS / 'stage5_efficiency_comparison.json').read_text())
pilot = json.loads((RESULTS / 'stage5_lora_pilot_result.json').read_text())
print('SMOKE TEST: PASSED AT BATCH 32')
print('TEST SPLIT LOADED:', pilot['test_split_loaded'])
display(comparison)
print('\nEFFICIENCY COMPARISON:')
print(json.dumps(efficiency, indent=2))
print('\nVALIDATION TRAJECTORY:')
display(pd.DataFrame(pilot['validation_evaluations']))
print('\nLOSS SUMMARY:')
print(json.dumps(pilot['loss_summary'], indent=2))
print('\nBEST ADAPTER:', pilot['saved_best_adapter'])
print('CORPUS CACHE:', DRIVE_ROOT / 'stage5_lora/pilot_seed42/embeddings')
